# Student Performance Prediction

Predict a student's final academic score using study-related and academic features.

**Dataset disclosure:** The included 1,000-row dataset is synthetic practice data generated for this portfolio project. It contains no real student records.

## Project goals
- Explore the dataset with charts.
- Prepare numerical and categorical features.
- Train Linear Regression and Random Forest models.
- Compare MAE, RMSE, and R².
- Review feature importance and make an example prediction.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42


## 2. Load and inspect data

In [ ]:
df = pd.read_csv("student_performance.csv")
print("Dataset shape:", df.shape)
display(df.head())
display(df.describe(include="all"))
print("\\nMissing values:")
print(df.isnull().sum())


## 3. Exploratory data analysis

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["final_score"], bins=20, edgecolor="black")
plt.title("Distribution of Final Scores")
plt.xlabel("Final score")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(df["study_hours_per_week"], df["final_score"], alpha=0.6)
plt.title("Study Hours vs Final Score")
plt.xlabel("Study hours per week")
plt.ylabel("Final score")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(df["attendance_percentage"], df["final_score"], alpha=0.6)
plt.title("Attendance vs Final Score")
plt.xlabel("Attendance (%)")
plt.ylabel("Final score")
plt.tight_layout()
plt.show()


## 4. Prepare features and target

In [ ]:
X = df.drop(columns=["final_score"])
y = df["final_score"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 5. Train and compare models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=8, random_state=RANDOM_STATE
    )
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })
    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
display(results_df.round(3))


## 6. Model performance chart

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(results_df["Model"], results_df["R2"])
plt.title("Model Performance Comparison (Test R²)")
plt.ylabel("R² score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## 7. Random Forest feature importance

In [ ]:
rf = trained_models["Random Forest"]
feature_names = rf.named_steps["preprocessor"].get_feature_names_out()
importances = rf.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({
    "Feature": feature_names, "Importance": importances
}).sort_values("Importance", ascending=False).head(10)
display(importance_df)

plt.figure(figsize=(9,5))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.gca().invert_yaxis()
plt.title("Top 10 Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 8. Example prediction

In [ ]:
sample = pd.DataFrame([{
    "study_hours_per_week": 10,
    "attendance_percentage": 90,
    "previous_score": 75,
    "absences": 3,
    "assignment_average": 78.0,
    "parent_support": "Medium",
    "internet_access": "Yes",
    "extracurricular_activities": "Yes"
}])

best_name = results_df.iloc[0]["Model"]
predicted_score = trained_models[best_name].predict(sample)[0]
print("Best model by test R²:", best_name)
print(f"Predicted final score: {predicted_score:.1f}")


## 9. Conclusion

Use the comparison table and charts generated above to describe model performance. Metrics are calculated when the notebook is run; report only the values from your own execution.

**Limitation:** This is a learning demonstration using synthetic data. Its generated patterns should not be used to make decisions about real students.
